# 01 · SQL data prep

**Goal:** load the raw Hillstrom CSV into DuckDB, clean it with saved SQL queries, and check data quality before any analysis.

All queries live in [`sql/`](../sql) and are run from Python with `hillstrom.data.run_sql_file`. The raw CSV in `data/raw/` is never modified; every cleaning step happens in SQL.

Why DuckDB: it reads the CSV directly with no database server to install, runs analytical SQL fast, and returns results as pandas DataFrames. Anyone who clones the repo can re-run everything with `pip install`.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from hillstrom.data import connect, run_sql_file
from hillstrom.paths import RAW_CSV, SQL_DIR

pd.set_option("display.float_format", "{:,.4f}".format)


def show_sql(name: str) -> None:
    # Render a saved query in the notebook so the SQL is visible next to its result.
    display(Markdown(f"**`sql/{name}`**\n```sql\n{(SQL_DIR / name).read_text()}\n```"))


con = connect(RAW_CSV)  # in-memory DuckDB; runs sql/01_load_clean.sql

## 1. Load and clean

The load query fixes the column types explicitly (no type guessing), adds a `customer_id` equal to the row position in the raw file, and adds two derived columns:

- `arm`: short labels for the three experiment groups (`mens`, `womens`, `control`).
- `purchase_group`: whether the customer bought men's merchandise, women's, or both in the past year. This is a **pre-treatment** variable, so it is safe to use for segment analysis later.

It also fixes a spelling quirk in the raw file: `zip_code` contains `Surburban`, recoded to `Suburban`.

In [2]:
show_sql("01_load_clean.sql")

**`sql/01_load_clean.sql`**
```sql
-- Load the raw Hillstrom CSV into a typed, cleaned `customers` table.
-- Parameter: $csv_path (path to the raw CSV). The raw file itself is never modified.
--
-- Cleaning steps:
--   * explicit column types instead of type sniffing
--   * customer_id = row position in the raw file (1-based)
--   * zip_code: recode the raw spelling 'Surburban' to 'Suburban'
--   * arm: short treatment labels used throughout the analysis
--   * purchase_group: past-year purchase history (mens only / womens only / both),
--     a pre-treatment variable used for segment analysis

CREATE OR REPLACE TABLE customers AS
WITH raw AS (
    SELECT *
    FROM read_csv(
        $csv_path,
        header = true,
        columns = {
            'recency': 'INTEGER',
            'history_segment': 'VARCHAR',
            'history': 'DOUBLE',
            'mens': 'INTEGER',
            'womens': 'INTEGER',
            'zip_code': 'VARCHAR',
            'newbie': 'INTEGER',
            'channel': 'VARCHAR',
            'segment': 'VARCHAR',
            'visit': 'INTEGER',
            'conversion': 'INTEGER',
            'spend': 'DOUBLE'
        }
    )
)
SELECT
    row_number() OVER () AS customer_id,
    recency,
    history_segment,
    history,
    mens,
    womens,
    CASE
        WHEN mens = 1 AND womens = 1 THEN 'both'
        WHEN mens = 1 THEN 'mens_only'
        WHEN womens = 1 THEN 'womens_only'
    END AS purchase_group,
    CASE WHEN zip_code = 'Surburban' THEN 'Suburban' ELSE zip_code END AS zip_code,
    newbie,
    channel,
    segment,
    CASE segment
        WHEN 'Mens E-Mail' THEN 'mens'
        WHEN 'Womens E-Mail' THEN 'womens'
        WHEN 'No E-Mail' THEN 'control'
    END AS arm,
    visit,
    conversion,
    spend
FROM raw;

```

In [3]:
con.sql("SELECT * FROM customers LIMIT 5").df()

,customer_id,recency,history_segment,history,mens,womens,purchase_group,zip_code,newbie,channel,segment,arm,visit,conversion,spend
0,1,10,2) $100 - $200,142.4400,1,0,mens_only,Suburban,0,Phone,Womens E-Mail,womens,0,0,0.0000
1,2,6,3) $200 - $350,329.0800,1,1,both,Rural,1,Web,No E-Mail,control,0,0,0.0000
2,3,7,2) $100 - $200,180.6500,0,1,womens_only,Suburban,1,Web,Womens E-Mail,womens,0,0,0.0000
3,4,9,5) $500 - $750,675.8300,1,0,mens_only,Rural,1,Web,Mens E-Mail,mens,0,0,0.0000
4,5,2,1) $0 - $100,45.3400,1,0,mens_only,Urban,0,Web,Womens E-Mail,womens,0,0,0.0000


In [4]:
# After cleaning, zip_code has three correctly spelled levels.
con.sql("SELECT zip_code, count(*) AS n FROM customers GROUP BY 1 ORDER BY 1").df()

,zip_code,n
0,Rural,9563
1,Suburban,28776
2,Urban,25661


## 2. Data quality checks

Before estimating anything, check that the data is internally consistent. Each column below counts rows that break a rule and should be zero:

- no missing values, every row mapped to an arm, unique ids, valid amounts;
- outcome logic: spend > 0 exactly when the customer converted, and every converter visited;
- every customer has some past-year purchase history (the population is past-year buyers).

In [5]:
show_sql("04_data_quality.sql")

**`sql/04_data_quality.sql`**
```sql
-- Data quality checks on the cleaned `customers` table (from 01_load_clean.sql).
-- Returns one row. Every column except n_rows counts violating rows and should be 0.

SELECT
    count(*) AS n_rows,
    count(*) FILTER (
        WHERE recency IS NULL OR history_segment IS NULL OR history IS NULL
           OR mens IS NULL OR womens IS NULL OR zip_code IS NULL OR newbie IS NULL
           OR channel IS NULL OR segment IS NULL OR visit IS NULL
           OR conversion IS NULL OR spend IS NULL
    ) AS rows_with_missing,
    count(*) FILTER (WHERE arm IS NULL) AS unmapped_arm,
    count(*) - count(DISTINCT customer_id) AS duplicate_ids,
    count(*) FILTER (WHERE spend < 0 OR history <= 0 OR recency < 1) AS invalid_amounts,
    -- Outcome logic: spend > 0 <=> converted, and converted => visited.
    count(*) FILTER (WHERE spend > 0 AND conversion = 0) AS spend_without_conversion,
    count(*) FILTER (WHERE spend = 0 AND conversion = 1) AS conversion_without_spend,
    count(*) FILTER (WHERE conversion = 1 AND visit = 0) AS conversion_without_visit,
    -- Population is past-year purchasers, so each must have bought mens or womens.
    count(*) FILTER (WHERE mens = 0 AND womens = 0) AS no_purchase_history
FROM customers;

```

In [6]:
quality = run_sql_file(con, "04_data_quality.sql")
quality.T.rename(columns={0: "count"})

,count
n_rows,64000
rows_with_missing,0
unmapped_arm,0
duplicate_ids,0
invalid_amounts,0
spend_without_conversion,0
conversion_without_spend,0
conversion_without_visit,0
no_purchase_history,0


**Result:** 64,000 rows and zero violations. The file matches its documentation, so no rows are dropped or imputed.